In [10]:
from pathlib import Path

cwd = Path.cwd().resolve()

for candidate in [cwd, *cwd.parents]:
    if (candidate / "data").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find the project root containing the 'data' folder.")

DATA_DIR = PROJECT_ROOT / "data"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)

PROJECT_ROOT: /home/rafidahmed/Coding/AI /rag_pipeline
DATA_DIR: /home/rafidahmed/Coding/AI /rag_pipeline/data


In [11]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [12]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Searching PDFs in: {pdf_dir}")
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data/pdf directory
all_pdf_documents = process_all_pdfs(DATA_DIR / "pdf")
print(f"all_pdf_documents contains {len(all_pdf_documents)} documents")

Searching PDFs in: /home/rafidahmed/Coding/AI /rag_pipeline/data/pdf
Found 1 PDF files to process

Processing: dp_presentation.pdf
  Loaded 19 pages

Total documents loaded: 19
all_pdf_documents contains 19 documents


In [ ]:
# split document into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],)
    
    all_chunks = []
    
    for doc in documents:
        try:
            chunks = text_splitter.split_text(doc.page_content)
            for i, chunk in enumerate(chunks):
                chunk_doc = doc.copy()
                chunk_doc.page_content = chunk
                chunk_doc.metadata['chunk_index'] = i
                all_chunks.append(chunk_doc)
            print(f"  ✓ Split document into {len(chunks)} chunks")
        except Exception as e:
            print(f"  ✗ Error splitting document: {e}")
    
    print(f"\nTotal chunks created: {len(all_chunks)}")
    return all_chunks

In [15]:
chunks = split_documents(all_pdf_documents)

  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks

Total chunks created: 19


/tmp/ipykernel_34691/3482337602.py:10: PydanticDeprecatedSince20: The `copy` method is deprecated; use `model_copy` instead. See the docstring of `BaseModel.copy` for details about how to handle `include` and `exclude`. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  chunk_doc = doc.copy()


In [16]:
# Show example chunks
print("Example chunks:")
for i, chunk in enumerate(chunks[:5]):  # Show first 5 chunks
    print(f"\nChunk {i+1} (from {chunk.metadata.get('source_file', 'unknown')}):")
    print(chunk.page_content[:300] + "..." if len(chunk.page_content) > 300 else chunk.page_content)
    print("-" * 50)

Example chunks:

Chunk 1 (from dp_presentation.pdf):
Introduction 
Many modern apps and speech-based systems collect
and process users’ voice data.
This voice data often contains personal or sensitive
information.
Without proper protection, attackers can misuse it to
steal identities or gain unauthorized access.
3
--------------------------------------------------

Chunk 2 (from dp_presentation.pdf):
Domain Introduction
Audio Anonymization:
Hides or masks the
speaker’s identity while
keeping the spoken
content clear and
understandable.
Audio Encryption:
Scrambles the entire
audio so it becomes
completely unreadable
without decryption.
Use Case: 
Enables safe sharing of
speech data for daily life...
--------------------------------------------------

Chunk 3 (from dp_presentation.pdf):
Literature
Review
5
--------------------------------------------------

Chunk 4 (from dp_presentation.pdf):
The VoicePrivacy 2022 Challenge: Progress &
Perspectives in Voice Anonymisation
Authors: Pierre 